# E4 — căn từng pha theo tổn thương của chính nó · build cache + train, một session

**Giả thuyết:** 8 pha **không khớp nhau**, và early-concat đang trộn mô không liên quan.

WORKLOG S-031 đã ĐO độ tán tâm tổn thương giữa 8 pha, trong toạ độ mm:

| Trục | Độ tán (trung vị) | Cửa sổ E3/E4 | Tỉ lệ |
|---|---|---|---|
| Trong mặt phẳng | 12,4 mm | ~53,8 mm | 23% |
| **Z** | **23,3 mm** | **~43,6 mm** | **53%** |

23,3mm đúng bằng biên độ chuyển động hô hấp của gan (10–25mm theo trục đầu-chân).
Ở hình học 112×112×32, đó là **17 trên 32 lát**.

Early-concat có tiền đề: voxel `(x,y,z)` của kênh `c` là **cùng một điểm giải phẫu** ở
mọi pha. Lệch 53% chiều sâu thì tiền đề vỡ. Hạng 2 của challenge thắng chính bằng
cách sửa registration.

## Vì sao E4 chứ không tiếp tục đổi hình học

Ba hình học đã thử, cùng một trần:

| | Hình học | macro-F1 (val fold 1) |
|---|---|---|
| E0 | cửa sổ cố định 144mm, 96×96×48 | 0,4244 |
| **E1** | lesion-tight, 96×96×48 | **0,5740** |
| E3 | lesion-tight, 112×112×32 | 0,5566 |

E3 − E1 = −0,017, nằm sâu trong nhiễu. Đổi tỉ lệ trục **không thay đổi gì**. Nếu bản
thân các pha không khớp thì đổi khung hình đúng là không giải quyết được — kết quả E3
nhất quán với giả thuyết này.

## E4 đổi ĐÚNG một biến so với E3

Hình học giữ nguyên 112×112×32. `configs/preprocess_e4.yaml` khác
`preprocess_e3.yaml` đúng hai khoá: `align_phases` và `cache_dir` (có test khoá điều đó).

> **E4 KHÔNG phải phép sửa trung tính.** Nó chỉ khử **tịnh tiến**, không khử xoay và
> biến dạng. Và **mô xung quanh sẽ thôi khớp** giữa các pha, chỉ tổn thương khớp. Với
> bài phân loại tổn thương thì có thể là điều mong muốn, nhưng đó là thay đổi ngữ
> nghĩa dữ liệu — phải vào limitations của báo cáo.

**Mount cần có:** dataset thô `lldmmridataset`. Không cần cache nào.

## 0. Bootstrap

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/hdtruong802/liver-mri-3d-classifier.git"
EXPERIMENT = "E4_per_phase"
CACHE_DIR_STR = "/kaggle/working/cache_e4"
PREPROCESS_CONFIG = "configs/preprocess_e4.yaml"

REPO = Path("/kaggle/working/repo")
os.chdir("/kaggle/working")
subprocess.run(["rm", "-rf", str(REPO)], check=False)
subprocess.run(["git", "clone", "-q", REPO_URL, str(REPO)], check=True)
sys.path.insert(0, str(REPO))
os.chdir(REPO)

# Python giữ module đã import trong sys.modules; clone lại KHÔNG tự làm mới (S-035).
for name in [m for m in list(sys.modules) if m == "src" or m.startswith("src.")]:
    del sys.modules[name]

print("repo commit:", subprocess.run(
    ["git", "-C", str(REPO), "log", "-1", "--format=%h %s"],
    capture_output=True, text=True,
).stdout.strip())

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "monai"], check=True)

os.environ["LLDMMRI_CACHE_DIR"] = CACHE_DIR_STR
os.environ["LLDMMRI_OUTPUT_DIR"] = f"/kaggle/working/runs/{EXPERIMENT}"

from src.utils.io import load_yaml, repo_root  # noqa: E402

assert repo_root() == REPO.resolve(), "src/ nạp từ chỗ khác — restart kernel"
cfg_e4 = load_yaml(REPO / PREPROCESS_CONFIG)
print("hình học    :", cfg_e4["target_size"], "(giữ nguyên như E3)")
print("align_phases:", cfg_e4["align_phases"])
assert cfg_e4["align_phases"] == "per_phase", "config này không phải E4"

# PHẦN A — Build cache

## Cổng A1 ⚠️ — thử 20 ca

In [ ]:
!rm -rf /kaggle/working/cache_e4
!python -m src.preprocess.build_cache --config configs/preprocess_e4.yaml --limit 20

In [ ]:
import pandas as pd

df = pd.read_csv(f"{CACHE_DIR_STR}/build_log.csv")
print("crop_source:", df.crop_source.value_counts().to_dict())
print("lỗi        :", (df.status != "ok").sum())
print("giây/ca    :", round(df.seconds.mean(), 2),
      f"-> cả 498 ca ~{df.seconds.mean() * 498 / 60:.0f} phút")

assert (df.status == "ok").all(), "có ca lỗi — xem build_log.csv"
assert (df.crop_source == "mask").mean() > 0.8, "phần lớn rơi về bbox, kiểm label_suffixes"
print("\nCổng A1 qua.")

## Cổng A2 ⚠️⚠️ — phép căn có THẬT SỰ làm gì không

**Đây là cổng quan trọng nhất của E4.** Nếu `max_shift_mm` toàn 0 thì phép căn không
có hiệu lực, cache E4 giống hệt cache E3, và train là phí 4 giờ GPU để ra lại đúng
0,5566.

Kỳ vọng: trung vị vài chục mm, khớp biên độ S-031 đã đo (12,4mm trong mặt phẳng,
23,3mm theo Z).

In [ ]:
print(df.max_shift_mm.describe().round(2).to_string())
print("\nsố pha phải rơi về tâm tham chiếu:", int(df.n_fallback_center.sum()),
      f"trên {len(df) * 8} pha")

assert df.max_shift_mm.max() > 0, (
    "max_shift_mm TOÀN 0 — phép căn không có hiệu lực. Cache này giống E3, "
    "đừng train. Kiểm align_phases trong config và cache_meta.json."
)
assert df.max_shift_mm.median() > 3.0, (
    f"trung vị độ dịch chỉ {df.max_shift_mm.median():.1f}mm, nhỏ hơn nhiều so với "
    "biên độ 12–23mm mà S-031 đo được. Nghi ngờ: bbox từng pha không được đọc đúng."
)
print("\nCổng A2 qua — phép căn có hiệu lực.")

## A3 — Build cả mẻ (~26 phút)

In [ ]:
!python -m src.preprocess.build_cache --config configs/preprocess_e4.yaml

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd

cache = Path(CACHE_DIR_STR)
meta = json.loads((cache / "cache_meta.json").read_text(encoding="utf-8"))
n_npz = len(list(cache.glob("*.npz")))

print("target_size :", meta["target_size"], "| align_phases:", meta.get("align_phases"))
print("số .npz     :", n_npz, "(cần 498)")
assert meta.get("align_phases") == "per_phase"

with np.load(next(cache.glob("MR*.npz"))) as d:
    print("shape một ca:", d["image"].shape)
    print("phase_shift_mm của ca này (mm, theo từng pha):")
    print(np.round(d["phase_shift_mm"], 1))
    assert d["image"].shape == (8, 112, 112, 32), f"hình học sai: {d['image'].shape}"

df = pd.read_csv(cache / "build_log.csv")
print("\n--- độ dịch trên cả 498 ca ---")
print(df.max_shift_mm.describe().round(2).to_string())
print("\npha rơi về tâm tham chiếu:", int(df.n_fallback_center.sum()), f"trên {len(df) * 8}")
print("lỗi:", (df.status != "ok").sum())
assert n_npz == 498, f"chỉ có {n_npz}/498"
print("\nCache E4 xong.")

# PHẦN B — Train fold 1

Dùng lại `configs/baseline_3dpatch.yaml` **không sửa một dòng** — giống E1 và E3.
Chỉ dữ liệu đổi.

## Cổng B1 — hợp đồng dữ liệu và model

In [ ]:
import torch
from torch.utils.data import DataLoader

from src.data.dataset import build_fold_datasets
from src.data.taxonomy import SHORT_NAMES
from src.models import build_model, count_parameters
from src.utils.io import resolve_cache_dir

CFG_PATH = REPO / "configs" / "baseline_3dpatch.yaml"
CFG = load_yaml(CFG_PATH)
CACHE_DIR = resolve_cache_dir(CFG)
SPLITS_DIR = repo_root() / CFG.get("splits_dir", "splits")
FOLD = CFG["fold"]

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", torch.cuda.get_device_name(0) if DEVICE.type == "cuda" else "CPU")
print("cache :", CACHE_DIR)

train_ds, val_ds = build_fold_datasets(CACHE_DIR, FOLD, splits_dir=SPLITS_DIR)
print(f"fold {FOLD}: train={len(train_ds)} val={len(val_ds)} (tổng phải = 394)")

model = build_model(CFG["model"]).to(DEVICE)
print(f"tham số: {count_parameters(model):,}")

batch = next(iter(DataLoader(train_ds, batch_size=2)))
images = batch["image"].to(DEVICE)
model.eval()
with torch.no_grad():
    logits = model(images)
print(f"\nvào {tuple(images.shape)} -> ra {tuple(logits.shape)}")
assert tuple(images.shape[1:]) == (8, 112, 112, 32), "dataset không trả hình học E3/E4"
assert logits.shape == (2, 7) and torch.isfinite(logits).all()
print("Cổng B1 qua.")

## Cổng B2 ⚠️ — đo thời gian

Hình học giống E3 nên kỳ vọng gần giống: E1 4,09h, E3 tương tự. Lệch nhiều là dấu
hiệu có gì khác đang xảy ra.

In [ ]:
import gc
import time

from src.train.loop import make_amp_scaler, run_epoch
from src.train.run import build_loaders, build_param_groups, build_scheduler
from src.utils.seed import set_seed

BUDGET_HOURS_PER_FOLD = 6.0
PROBE_EPOCHS = 2

TCFG, DCFG = CFG["train"], CFG["data"]
set_seed(CFG["seed"])

probe_train_loader, probe_val_loader, probe_labels = build_loaders(CFG, FOLD)
probe_model = build_model(CFG["model"]).to(DEVICE)
probe_opt = torch.optim.AdamW(
    build_param_groups(probe_model, float(TCFG["weight_decay"])), lr=float(TCFG["lr"])
)
probe_sched = build_scheduler(probe_opt, TCFG, int(TCFG["epochs"]))
probe_amp = bool(TCFG.get("amp", True)) and DEVICE.type == "cuda"
probe_scaler = make_amp_scaler(probe_amp)
criterion = torch.nn.CrossEntropyLoss()
accum = int(TCFG["accum_steps"])

timings = []
for i in range(PROBE_EPOCHS):
    t0 = time.time()
    tr = run_epoch(probe_model, probe_train_loader, DEVICE, criterion,
                   optimizer=probe_opt, scaler=probe_scaler, accum_steps=accum, amp=probe_amp)
    va = run_epoch(probe_model, probe_val_loader, DEVICE, criterion, amp=probe_amp)
    probe_sched.step()
    timings.append(time.time() - t0)
    print(f"epoch thử {i+1}: {timings[-1]:.1f}s | train {tr['loss']:.4f} | val {va['loss']:.4f}")

per_epoch = timings[-1]
hours_one = per_epoch * int(TCFG["epochs"]) / 3600
print(f"\n~{per_epoch:.1f}s/epoch × {TCFG['epochs']} = **{hours_one:.2f} giờ/fold**")
print("(E1: 4.09h — E4 nên xấp xỉ vì hình học giống E3)")

del probe_model, probe_opt, probe_sched, probe_scaler
gc.collect()
if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

if hours_one > BUDGET_HOURS_PER_FOLD:
    raise RuntimeError(f"{hours_one:.2f} giờ/fold, vượt ngân sách {BUDGET_HOURS_PER_FOLD} giờ.")
print("Cổng B2 qua.")

## B3 — Train (chạy lại chính cell này nếu session bị ngắt, `resume: true`)

In [ ]:
from src.train.run import train

result = train(CFG_PATH, fold_override=FOLD)
print(result)

## B4 — Kết quả và phán quyết

In [ ]:
import json

import numpy as np

from src.train.run import run_dir

RUN_DIR = run_dir(CFG, FOLD)
print("run dir:", RUN_DIR)
best = json.loads((RUN_DIR / "metrics_best.json").read_text(encoding="utf-8"))

print(f"\nfold {best['fold']} · epoch {best['epoch']} · seed {best['seed']}")
for key in ("macro_f1", "balanced_accuracy", "accuracy", "cohen_kappa"):
    print(f"  {key:>18}: {best[key]:.4f}")

print("\nF1 từng lớp:")
for k, f1 in enumerate(best["per_class_f1"]):
    print(f"  {SHORT_NAMES[k]:>7}: {f1:.3f}")

print("\nMa trận nhầm lẫn (hàng = thật, cột = đoán):")
print("        " + "".join(f"{SHORT_NAMES[k]:>8}" for k in sorted(SHORT_NAMES)))
for k, row in enumerate(np.array(best["confusion_matrix"])):
    print(f"{SHORT_NAMES[k]:>7} " + "".join(f"{v:>8d}" for v in row))

f1 = best["macro_f1"]
print("\n--- ĐỐI CHIẾU (cùng fold 1, 82 ca val, cùng config train) ---")
print("  E0  144mm cố định, 96×96×48        : 0.4244")
print("  E1  lesion-tight, 96×96×48         : 0.5740")
print("  E3  lesion-tight, 112×112×32       : 0.5566")
print(f"  E4  + căn từng pha                 : {f1:.4f}   <- lần này")
print(f"\n  E4 - E3 = {f1 - 0.5566:+.4f}   (cùng hình học, chỉ khác phép căn)")
print(f"  E4 - E1 = {f1 - 0.5740:+.4f}   (mốc tốt nhất tới giờ)")

print("\n--- PHÁN QUYẾT, luật chốt trước ở WORKLOG S-067/S-069 ---")
if f1 >= 0.62:
    print("  >= 0.62: misalignment ĐÚNG là nút thắt. Đi tiếp:")
    print("           rigid registration thật, rồi quay lại Siamese ở hình học đúng.")
elif f1 >= 0.5740 + 0.03:
    print("  Cải thiện rõ nhưng chưa tới 0.62: giữ per_phase làm mặc định,")
    print("           cân nhắc rigid registration để khử cả xoay/biến dạng.")
else:
    print("  KHÔNG cải thiện: DỪNG TÌM KIẾM, chuyển sang TÁI LẬP.")
    print("           Dựng đúng ResNet3D @ 14x112x112 + Focal loss của CGHNet Bảng 1,")
    print("           xem có ra ~0.709 không. Ra được thì pipeline lành và ta chỉ chọn")
    print("           sai cấu hình; không ra được thì CÓ LỖI TRONG PIPELINE, và mọi giờ")
    print("           GPU tiêu vào tìm kiếm kiến trúc từ đó là vô ích.")

print("\n  ⚠️ val fold 1 (82 ca) KHÔNG so trực tiếp được với test-104 của văn liệu.")
print("     CI ở n=82 rộng ~±0.10 -> đây là SÀNG LỌC, không phải số báo cáo.")

### Đường cong overfitting

In [ ]:
import pandas as pd

log = pd.read_csv(RUN_DIR / "train_log.csv")
i, j = log.val_loss.idxmin(), log.val_macro_f1.idxmax()
last = log.iloc[-1]
print(f"val_loss đáy : epoch {int(log.epoch[i]):3d} = {log.val_loss[i]:.3f}"
      "   (E0 ep10=1.767 · E1 ep9=1.734)")
print(f"F1 đỉnh      : epoch {int(log.epoch[j]):3d} = {log.val_macro_f1[j]:.4f}"
      "  (E0 ep162 · E1 ep200 · E3 ep145)")
print(f"gap cuối     : {last.val_loss - last.train_loss:+.3f}         (E0 +2.838 · E1 +2.547)")
print(f"thời gian    : {log.seconds.sum() / 3600:.2f}h")

### Calibration và selective — `T` cross-fit 5 phần

In [ ]:
import numpy as np

from src.eval import calibration as C
from src.eval import selective as S
from src.eval.bootstrap import bootstrap_metric
from src.eval.metrics import macro_f1

d = np.load(RUN_DIR / "val_probs_best.npz", allow_pickle=True)
probs = d["probs"].astype(np.float64)
probs /= probs.sum(1, keepdims=True)
labels = d["labels"]
pred, conf = probs.argmax(1), probs.max(1)
correct = (pred == labels).astype(int)


def crossfit_T(p, y, k=5, seed=1337):
    idx = np.random.default_rng(seed).permutation(len(y))
    out, temps = np.empty_like(p), []
    for i in range(k):
        te = idx[i::k]
        tr = np.setdiff1d(idx, te)
        t = C.fit_temperature(p[tr], y[tr])
        temps.append(t)
        out[te] = C.apply_temperature(p[te], t)
    return out, np.array(temps)


cal, temps = crossfit_T(probs, labels)
ci = bootstrap_metric(labels, pred, macro_f1, n_resamples=4000)
print(f"macro-F1 {ci['point']:.4f} [{ci['ci_low']:.4f}, {ci['ci_high']:.4f}]")
print(f"ECE  {C.expected_calibration_error(probs, labels):.4f}"
      f" -> {C.expected_calibration_error(cal, labels):.4f}")
print(f"NLL  {C.negative_log_likelihood(probs, labels):.4f}"
      f" -> {C.negative_log_likelihood(cal, labels):.4f}  (đoán mò {np.log(7):.4f})")
print(f"T    {temps.mean():.3f} ± {temps.std():.3f}")
print(f"AURC {S.aurc(correct, conf):.4f}  (thấp là tốt)")
print("\naccuracy theo coverage:")
for cov in (1.0, 0.9, 0.8, 0.7):
    print(f"   {cov:4.0%}  {S.selective_accuracy(correct, conf, cov):.4f}")

print("\n--- ĐỐI CHIẾU ---")
print("  E0: ECE 0.3218 -> 0.1455 | NLL 2.7172 -> 1.7251 | AURC 0.5395 | T 4.150")
print("  E1: ECE 0.2935 -> 0.2505 | NLL 3.3182 -> 1.5205 | AURC 0.2753 | T 5.010")
print("\n⚠️ Đừng báo macro-F1@coverage trên một fold (S-060).")

## B5 — Giữ lại gì

Cache E4 ở `/kaggle/working/cache_e4` (~3,2 GB) — **Save Version** nếu muốn dùng lại
mà khỏi build 26 phút.

In [ ]:
import shutil
from pathlib import Path

out = Path(f"/kaggle/working/{EXPERIMENT}_results")
out.mkdir(exist_ok=True)
for name in ["val_probs_best.npz", "metrics_best.json", "train_log.csv", "config_used.json"]:
    shutil.copy(RUN_DIR / name, out / name)
shutil.copy(Path(CACHE_DIR_STR) / "cache_meta.json", out / "cache_meta.json")
# Log build đi kèm: nó chứa độ dịch từng ca, tức bằng chứng phép căn đã làm gì.
shutil.copy(Path(CACHE_DIR_STR) / "build_log.csv", out / "cache_build_log.csv")
shutil.make_archive(str(out), "zip", out)
print("đã gói:", round(Path(f"{out}.zip").stat().st_size / 2**10, 1), "KiB")

for f in sorted(RUN_DIR.iterdir()):
    print(f"   {f.name:24s} {f.stat().st_size / 2**20:8.1f} MiB")

from IPython.display import FileLink

FileLink(f"{EXPERIMENT}_results.zip")